> **Open in Google Colab.** The next cell installs `coptpy`. The bundled free license is size-limited but covers this case. Then run *Runtime > Run all*.


In [ ]:
# COPT Python API and this case's dependencies
%pip install -q coptpy


# After-class exercises-code implementation

The test data is consistent with the original model.

In [1]:
from coptpy import *
# Create environment
env = Envr()

# Create model 
model = env.createModel(name="course_scheduling_exercise")

instructor, instructorRating, instructorTime = multidict({
    ('MGT 490', 1): [4.3, [('Mon', 4)]],
    ('MGT 490', 2): [3.8, [('Tues', 4)]],
    ('MGT 490', 3): [3.5, [('Wed', 4)]],
    ('MGT 490', 4): [3.5, [('Fri', 4)]],
    ('MGT 490', 5): [4.6, [('Mon', 1), ('Wed', 2)]],
    ('MGT 490', 6): [2.7, [('Tues', 2), ('Thur', 1)]],
    ('FIN 358', 1): [3.5, [('Wed', 4)]],
    ('FIN 358', 2): [3.3, [('Tues', 2), ('Thur', 1)]],
    ('CIS 102T', 1): [4.4, [('Wed', 3)]],
    ('CIS 102T', 2): [3.1, [('Thur', 3)]],
    ('CIS 102W', 1): [3.7, [('Tues', 4)]],
    ('CIS 102W', 2): [3.5, [('Wed', 3)]],
    ('FIN 325', 1): [3.0, [('Thur', 4)]],
    ('FIN 325', 2): [3.7, [('Mon', 1), ('Wed', 2)]],
    ('FIN 352', 1): [3.6, [('Mon', 4)]],
    ('FIN 352', 2): [3.9, [('Mon', 2), ('Wed', 1)]],
    ('FIN 356', 1): [3.2, [('Tues', 4)]],
    ('FIN 356', 2): [3.4, [('Tues', 2), ('Thur', 1)]],
    ('FIN 359', 1): [3.0, [('Mon', 4)]],
    ('FIN 359', 2): [3.5, [('Wed', 4)]],
}) 

weekdays = ['Mon', 'Tues', 'Wed', 'Thur', 'Fri']
timeInstructor = dict()
for w in weekdays:
    for j in [1,2,3,4]:
        timeInstructor[(w, j)] = []
for i in instructor:
    for t in instructorTime[i]:
        timeInstructor[t].append(i)
        
compulsoryCourse = ['MGT 490', 'FIN 358']
cisOptionalCourse = ['CIS 102T', 'CIS 102W']
finOptionalCourse = ['FIN 325', 'FIN 352', 'FIN 356', 'FIN 359']
optionalCourse = cisOptionalCourse + finOptionalCourse

Cardinal Optimizer v8.0.5. Build date May 30 2026
Copyright Cardinal Operations 2026. All Rights Reserved



## Build model

**<big><font color="#3c70c6">New variable</font></big>**

We have added a new 0-1 variable $y_k$ to indicate whether there are courses scheduled for week $k$.

In [3]:
x = model.addVars(instructor, vtype=COPT.BINARY)
y = model.addVars(weekdays, vtype=COPT.BINARY)

The objective function and constraints in the original model remain unchanged.

In [5]:
model.setObjective(quicksum(instructorRating[i] * x[i] for i in instructor), sense=COPT.MAXIMIZE)

model.addConstrs(quicksum(x[i] for i in instructor if i[0] == c) == 1 for c in compulsoryCourse)
model.addConstrs(quicksum(x[i] for i in instructor if i[0] == c) <= 1 for c in optionalCourse) 
model.addConstr(quicksum(x[i] for i in instructor for c in cisOptionalCourse if i[0] == c) == 1)
model.addConstr(quicksum(x[i] for i in instructor for c in finOptionalCourse if i[0] == c) == 2)

model.addConstrs(quicksum(x[i] for i in timeInstructor[t]) <= 1 for t in timeInstructor.keys())
model.addConstrs(quicksum(x[i] for i in timeInstructor[(k,1)] + timeInstructor[(k,2)]) <= 1 for k in weekdays)
model.addConstrs(quicksum(x[i] for i in timeInstructor[(k,2)] + timeInstructor[(k,3)]) <= 1 for k in weekdays)

**<big><font color="#3c70c6">Add new constraints</font></big>**

1. We have added a new constraint on arranging courses for up to three days per week:

In [7]:
model.addConstr(quicksum(y[j] for j in weekdays) <= 3)

<coptpy.Constraint: >

2. And the constraints on the relationship between daily course schedule and course selection decision-making for that day:

In [9]:
model.addConstrs(y[j] >= x[i] for j in weekdays for k in [1,2,3,4] for i in timeInstructor[(j,k)])

## Solve and obtain results

Printing the solution results, we found that "FIN 356｜3.4" in the original curriculum was replaced by "FIN 356｜3.2". The original optimal solution has changed, and the optimal objective function value dropped to 19.3.

In [11]:
model.solve()

timeCode = {
    1: "1:25-2:20 p.m.",
    2: "1:25-3:15 p.m.",
    3: "2:30-5:15 p.m.",
    4: "6:00-8:45 p.m."
}

if model.status == COPT.OPTIMAL:
    print("-"*58)
    print("|     course\t |     rating\t |         time\t\t |")
    for i in instructor:
        if x[i].x >= 0.9:
            timeList = instructorTime[i]
            print("-"*58)
            print("|    {0}\t |      {1}\t | {2}  {3}\t |".format(i[0], instructorRating[i], timeList[0][0], timeCode[timeList[0][1]]))
            if len(timeList) > 1:
                print("|\t\t |\t\t | {2}  {3}\t |".format(i[0], instructorRating[i], timeList[1][0], timeCode[timeList[1][1]]))
    print("-"*58)

Model fingerprint: 1db537a8

Using Cardinal Optimizer v8.0.5 on Windows (22H2 Build 19045 - x86_64)
The CPU model is Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz
Hardware has 4 physical cores and 8 logical cores. Using instruction set X86_AVX2 (10)
Maximizing a MIP problem

The original problem has:
    67 rows, 25 columns and 136 non-zero elements
    25 binaries

Starting the MIP solver with 8 threads and 32 tasks

Presolving the problem

The presolved problem has:
    11 rows, 13 columns and 49 non-zero elements
    13 binaries

Problem info:
    Range of matrix coefficients:    [1e+00,1e+00]
    Range of rhs coefficients:       [1e+00,3e+00]
    Range of bound coefficients:     [1e+00,1e+00]
    Range of cost coefficients:      [6e-01,4e+00]
    Density of cost:                     92.3%

     Nodes    Active  LPit/n  IntInf     BestBound  BestSolution     Gap   Time
         0         1      --       0  4.300000e+01            --     Inf  0.10s
H        0         1      --       0  4